# Bài 28 — chuỗi luyện Easy · LeetCode #141 — Linked List Cycle

**Mức độ:** Easy · **Ngôn ngữ:** Python 3 · **Đề gốc:** https://leetcode.com/problems/linked-list-cycle/

**Mục tiêu học tập**

- Đọc đề và tự phân biệt input/output, constraints và ví dụ biên.
- Nắm cú pháp Python theo từng cell thực nghiệm trước khi ghép thuật toán.
- Hiểu brute force/baseline trước, sau đó giải thích mấu chốt tối ưu bằng bất biến.
- Trace từng bước với dữ liệu dễ sai, không xem code như một hộp đen.
- Chạy lời giải chuẩn LeetCode, test mẫu, edge case, randomized/exhaustive khi phù hợp.
- Tự giải **6 bài luyện từ dễ đến khó**, chỉ xem gợi ý/đáp án cuối sau khi thử.

**Cách học:** đọc Markdown → tự dự đoán output → chạy cell Python ngay sau đó → giải thích lại bằng lời. Các cell TODO không chạy tự động và chỉ phục vụ tự luyện.

## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Cho `head` của linked list. Xác định có tồn tại chu trình nào khiến đi theo `.next` rồi quay về một node **đã gặp** không. Chỉ xét **node**, không xét giá trị trùng.

### Ví dụ quan trọng

Danh sách `3→2→0→-4`, trong đó `-4.next` trỏ về node `2` → True. `1→2→None` → False.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [ ]:
from __future__ import annotations
from typing import Optional

class ListNode:
    def __init__(self, val: int = 0, next=None):
        self.val, self.next = val, next
    def __repr__(self):
        return f"ListNode({self.val})"

def linked(values):
    dummy=ListNode(); tail=dummy
    for x in values:
        tail.next=ListNode(x); tail=tail.next
    return dummy.next

def node_values(head, limit=30):
    result=[]; cur=head; seen=set()
    while cur is not None and len(result)<limit:
        if id(cur) in seen:
            return result + ['(chu trình)']
        seen.add(id(cur)); result.append(cur.val); cur=cur.next
    if cur is not None: result.append('...')
    return result

### 2.1. Linked List là chuỗi nút nối bằng `next`

*Singly Linked List* (danh sách liên kết đơn) gồm các nút `ListNode`.
Mỗi nút có `.val` (dữ liệu) và `.next` (tham chiếu tới nút kế tiếp).
Gốc danh sách gọi là `head`; `head` cũng có thể là `None` khi danh sách rỗng.
Khác list Python, truy cập nút thứ i cần theo `.next` liên tiếp từ đầu.
Ví dụ `a -> b -> c -> None`: khi đứng tại a, lấy a.next mới đến b.
Trong bài có chu trình, `.next` của một nút cuối có thể quay lại nút trước.

In [ ]:
aa=ListNode(10)
bb=ListNode(20)
cc=ListNode(30)
aa.next=bb; bb.next=cc
print('head:',aa.val,'kế tiếp:',aa.next.val,'kế nữa:',aa.next.next.val)
print('nút cuối trỏ vào:',cc.next)
assert aa.next is bb and cc.next is None

### 2.2. Tham chiếu khác giá trị; `is` khác `==`

Hai nút có thể cùng `.val == 7` nhưng là **hai object** khác nhau.
Chu trình xảy ra nếu đi theo `.next` và gặp lại **chính object đã đi qua**;
không phải cứ thấy số 7 lần hai là có cycle.
Viết `slow is fast` để kiểm tra cùng tham chiếu, không dùng `slow.val == fast.val`.
`a = b` cho a và b cùng trỏ đối tượng; không copy node.
Đây là lý do baseline dùng `set()` các **node**, không phải set các giá trị.
Hãy thử ví dụ có hai nút cùng giá trị nhưng không có chu trình.

In [ ]:
first=ListNode(7)
second=ListNode(7)
first.next=second
print('giá trị bằng?',first.val==second.val)
print('cùng nút?',first is second)
assert first is not second and second.next is None

### 2.3. `None` và vì sao điều kiện `fast and fast.next` bắt buộc

Một nút `None` không có thuộc tính `.next`. Nếu viết `fast.next.next`
khi `fast` hoặc `fast.next` bằng None, Python gây `AttributeError`.
Với `while fast is not None and fast.next is not None`,
điều kiện sau chỉ chạy nếu điều kiện trước đúng nhờ **short-circuit**.
Chuỗi kết thúc ở None nghĩa là không có cycle; lúc đó `fast` chạy tới cuối.
Không thể kiểm tra `slow is fast` ở trước lần di chuyển đầu tiên vì
hai con trỏ đều khởi tạo tại head và sẽ cho **dương tính giả** ở danh sách bình thường.

In [ ]:
head_demo=ListNode(3)
fast_demo=head_demo
print('an toàn để nhảy 2 bước?',fast_demo is not None and fast_demo.next is not None)
fast_demo=fast_demo.next
print('đã đến cuối?',fast_demo is None)
assert fast_demo is None

### 2.4. `set` lưu nút đã gặp: baseline dễ hiểu

`seen=set()` là tập hợp các phần tử **không trùng nhau**.
Có thể hỏi `node in seen` để biết object đó đã từng được lưu chưa,
rồi dùng `seen.add(node)` để thêm node mới.
Nếu duyệt đến một nút đã xuất hiện → có cycle; nếu duyệt tới None → không có.
Đây là cách rất dễ nghĩ và phù hợp làm **oracle** (đáp án đối chiếu) khi test.
Trong notebook, `ListNode` là class đơn giản, hash theo identity;
nếu bạn tự viết dataclass có `__eq__` mà không có `__hash__`, set có thể không chấp nhận object.
Dùng set tốn O(n) extra space nên ta cần tìm cách khác.

In [ ]:
p=ListNode(1);q=ListNode(2);r=ListNode(3)
p.next=q;q.next=r
seen_nodes=set()
for node in (p,q,r,p):
    print('node',node,'đã thấy?',node in seen_nodes)
    seen_nodes.add(node)
assert len(seen_nodes)==3

### 2.5. Hai con trỏ có vận tốc khác nhau

**Slow/Fast Pointers** dùng hai biến đều trỏ tới node.
Trong mỗi vòng, `slow = slow.next` đi 1 cạnh; `fast = fast.next.next` đi 2 cạnh.
Nếu đường đi có đoạn cuối `None`, fast sẽ tới cuối trước hoặc cùng slow.
Nếu đường đi tạo vòng, fast sẽ thu hẹp chênh lệch vị trí so với slow trong vòng,
rồi cuối cùng bắt kịp theo modulo chiều dài vòng.
Không tạo node, không sửa `.next`; chỉ thay tham chiếu của hai biến.
Phải phân biệt quãng đường đi và vị trí object hiện tại.

In [ ]:
labels=list('ABCDEFG')
slow_index=0;fast_index=0
for step in range(1,4):
    slow_index+=1;fast_index+=2
    print('bước',step,'slow:',labels[slow_index],'fast:',labels[fast_index])
assert slow_index==3 and fast_index==6

### 2.6. Chu trình không có nghĩa giá trị nút phải lặp

`pos` trong ví dụ LeetCode chỉ **mô tả cách input được dựng**:
nếu `pos = 1` thì node cuối trỏ lại **node có index 1**.
Hàm submit chỉ nhận `head`, **không nhận tham số `pos`**.
Hai node liên tiếp có cùng `.val` mà `next` vẫn đi tới None không có cycle.
Ngược lại, một vòng có toàn giá trị khác nhau vẫn là cycle.
Để minh họa vòng, ta phải nối **lại object cũ** thay vì tạo node mới cùng giá trị.
Đừng gọi hàm in toàn bộ linked list nếu chưa có giới hạn bước, vì cycle gây lặp vô hạn.

In [ ]:
no_cycle=linked([7,7,7])
a=ListNode(1);b=ListNode(2);c=ListNode(3)
a.next=b;b.next=c;c.next=b
print('trùng value, không vòng:',node_values(no_cycle))
print('value khác, có vòng:',node_values(a,limit=8))
assert no_cycle.next is not no_cycle and c.next is b

### 2.7. Khoảng cách modulo: vì sao fast chắc chắn gặp slow?

Giả sử slow và fast đã cùng ở trong vòng độ dài `L`. Vì fast đi 2 nút,
slow đi 1 nút mỗi vòng lặp, vị trí tương đối của fast so slow thay đổi `+1` modulo L.
Sau nhiều nhất L lần như vậy, chênh lệch trở thành 0, hai con trỏ trỏ cùng object.
Chúng có thể gặp **không phải** tại điểm bắt đầu của chu trình; bài #141 chỉ hỏi True/False.
Với L=4, các chênh lệch 1→2→3→0 minh họa ý tưởng toán học.
Trường hợp vòng chỉ có **1 nút** cũng luôn gặp lại sau lần nhảy đầu tiên.

In [ ]:
cycle_size=4
relative=1
for turn in range(1,5):
    print('lượt',turn,'khoảng cách trước:',relative)
    relative=(relative+1)%cycle_size
    print('  khoảng cách sau:',relative)
assert relative==1

## 3. Python nền tảng — công cụ cần hiểu trước khi giải

`is` kiểm tra **hai biến có trỏ đúng một object** không; `==` hoặc `.val` không thay thế được. Con trỏ `.next` là tham chiếu, có thể quay về node cũ.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [ ]:
def cycle_list(values,pos):
    head=linked(values);nodes=[];p=head
    while p is not None:nodes.append(p);p=p.next
    if nodes and pos>=0:nodes[-1].next=nodes[pos]
    return head
head=cycle_list([3,2,0,-4],1)
print('ba node đầu:',head,head.next,head.next.next)
print('node cuối -> node thứ 2?',head.next.next.next.next is head.next)

### Câu hỏi gợi mở trước khi tối ưu

Sau khi tự chạy ví dụ nền tảng, hãy tự trả lời bằng lời:

1. Trong bài **Linked List Cycle**, đâu là dữ liệu thật, đâu là công cụ dựng thử?
2. Biến nào thay đổi sau một vòng lặp / sau một lời gọi đệ quy?
3. Khi nào một ký tự / node / giá trị được xem là 'đã xử lý xong'?
4. Nếu input rỗng, có một phần tử, hay chỉ có một nhánh, cần return kiểu gì?
5. Thao tác nào tăng theo số phần tử n, thao tác nào chỉ giữ vài biến?

Không cần trả lời bằng công thức ngay; thử input nhỏ rồi diễn đạt kết quả.

### Quy trình chạy notebook không phụ thuộc trạng thái ngầm

Các hàm dựng input, baseline, `solve`, trace và đáp án được định nghĩa theo thứ tự từ trên xuống dưới.
Không chạy phần đáp án trước phần bài tập nếu muốn tự luyện.
Khi thử một biến thể, hãy tạo input mới, không sửa input đã được dùng cho `assert` ở các cell phía trên.
Nếu chạy lại một cell, bạn nên kiểm tra các biến demo bị ghi đè; cách ổn định nhất là **Restart Kernel & Run All**.

Ở các bài Linked List/Tree, LeetCode đã cung cấp sẵn lớp node. Trong notebook, lớp node được định nghĩa để mô phỏng local và không phải mã cần nộp.

### 3.1. Dựng danh sách có/không có cycle bằng `cycle_list`

Trong LeetCode ví dụ có thể viết `[3,2,0,-4], pos=1`.
`pos` chỉ cho biết node **cuối** được nối về node ở vị trí 1.
Hàm `cycle_list(values,pos)` của notebook sẽ tạo node thật từ mỗi giá trị,
rồi nối tham chiếu `next` của node cuối trở lại object đã có.
Với `pos=-1`, không nối trở lại; danh sách kết thúc ở None.
Khi submit, không gọi hàm dựng dữ liệu này: bài chính **chỉ nhận `head`**.
Hàm `node_values` có giới hạn và ghi chú cycle để không in vô hạn.

In [ ]:
for values,pos in [([3,2,0,-4],1),([1,2],0),([7,7,7],-1),([], -1)]:
    head0=cycle_list(values,pos)
    print('input:',values,'pos:',pos,'đọc an toàn:',node_values(head0,limit=10))
assert node_values(cycle_list([1,2],0))==[1,2,'(chu trình)']

### 3.2. Đi từng liên kết theo `next`, đừng biến cây thành list Python

Bài toán ở đây không có chỉ số truy cập trực tiếp như `nums[i]`.
Để tới node kế tiếp, phải theo `current = current.next`.
Khi gặp cycle, vòng `while current is not None:` có thể **không bao giờ kết thúc**.
Vì vậy mọi hàm demo đi qua danh sách chưa biết có vòng hay không phải có giới hạn bước,
hoặc `seen` theo identity. Đó là tính an toàn khi giảng, không phải một phần code submit.
Thử in tối đa 6 nút của danh sách 1→2→3→2→3→... mà không treo kernel.

In [ ]:
linked_cycle=cycle_list([1,2,3],1)
pointer=linked_cycle
for step in range(6):
    print('bước',step,'node:',pointer,'identity:',id(pointer))
    pointer=pointer.next
assert pointer is not None

## 4. Cách nghĩ trực giác (baseline)

Dùng `set` lưu **tham chiếu node** đã đi qua. Gặp node lặp lại thì có chu trình; gặp None thì không. O(n) bộ nhớ.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [ ]:
def baseline(head):
    seen=set()
    while head is not None:
        if head in seen:return True
        seen.add(head);head=head.next
    return False

In [ ]:
for pos in [1,-1,0]:
    h=cycle_list([3,2,0,-4],pos);print('chu trình pos=',pos,'?',baseline(h))

### 4.1. Baseline dùng set phát hiện nút đã gặp lại

Baseline có một invariant đơn giản: `seen` lưu tất cả **node object** đã đi qua.
Khi gặp `node in seen`, chứng tỏ đã đi một vòng từ node đó quay trở lại,
nên chắc chắn có cycle. Nếu `node` trở thành None, danh sách hữu hạn không vòng.
Bởi vì mỗi node trong danh sách không vòng được thăm một lần,
time O(n) trung bình, memory O(n). Việc hash object của class ListNode
trong notebook dùng identity mặc định, không hash giá trị `.val`.
Đây là cơ sở đáng tin cậy để đối chiếu thuật toán Floyd.

In [ ]:
for pos in (0,1,-1):
    head0=cycle_list([1,2,3],pos)
    print('pos:',pos,'baseline:',baseline(head0),'dãy đọc:',node_values(head0,8))
    assert baseline(head0)==(pos!=-1)

### 4.2. Vì sao `seen_values` là sai: giá trị trùng không tạo cycle

Một cách làm sai là thêm `node.val` vào set thay vì chính node.
Với danh sách 1→2→1→None, giá trị 1 xuất hiện hai lần nhưng node đầu và
node thứ ba hoàn toàn khác nhau: **không có cycle**.
Nếu dùng set giá trị, thuật toán báo True sai.
Đây là khác biệt cốt lõi khi làm bài đồ thị / danh sách liên kết:
*node identity* và *payload value* không cùng ý nghĩa.
Ta viết hàm sai minh họa, không dùng trong lời giải cuối.

In [ ]:
def wrong_value_cycle(head):
    seen_values=set()
    while head is not None:
        if head.val in seen_values:return True
        seen_values.add(head.val)
        head=head.next
    return False
h=linked([1,2,1])
print('dùng value:',wrong_value_cycle(h),'dùng identity:',baseline(h))
assert wrong_value_cycle(h) and not baseline(h)

## 5. Vì sao cần chuyển sang kỹ thuật chính?

**Floyd two pointers**: con trỏ `slow` mỗi lần đi 1 node, `fast` mỗi lần đi 2 node. Nếu có vòng, fast sẽ đuổi kịp slow; nếu không, fast gặp None.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

### Bước đột phá — Không cần nhớ tất cả node, chỉ cần hai vận tốc

Set đúng nhưng tốn bộ nhớ tỉ lệ số node. Hãy tưởng tượng hai người chạy trên một tuyến đường:
- Tuyến đường thẳng có điểm cuối: người nhanh tới điểm cuối rồi phải dừng.
- Đường có một vòng tròn: người nhanh sẽ bắt kịp người chậm ở **cùng vị trí**.

Từ đó dùng `slow` (mỗi lượt 1 bước) và `fast` (mỗi lượt 2 bước).
**Quan trọng:** phải di chuyển cả hai rồi mới kiểm tra `slow is fast`;
nếu so ngay trước lần đầu, cả hai đều là `head` → luôn False Positive.

**Bất biến:** tại đầu mỗi vòng, cả hai biến đều là các nút nằm trên đường đi theo `.next`
từ `head`, hoặc `fast` đã chạm cuối. Không có phép nào thay đổi cấu trúc `.next`.
Nếu có vòng, khoảng cách tương đối trong vòng tăng 1 modulo L mỗi vòng, dẫn tới gặp nhau.
Nếu không có vòng, fast rơi vào None. Đây là nền tảng chứng minh Floyd.

## 6. Lời giải chính — hiểu từng bước rồi mới viết

`slow=head,fast=head`. Khi `fast` và `fast.next` tồn tại: slow=slow.next; fast=fast.next.next; nếu hai con trỏ `is` nhau, trả True; ra khỏi while thì False.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

### 5.1. Mổ xẻ điều kiện `while fast and fast.next`

Mỗi lượt fast đi hai cạnh, nên trước khi làm vậy phải chắc rằng
cả `fast` lẫn `fast.next` đều là node thật.
Điều kiện `while fast is not None and fast.next is not None` bảo đảm an toàn;
Python chỉ kiểm vế sau khi vế trước đúng (*short-circuit*).
Khi kết thúc do fast hoặc fast.next là None, đường đi tới cuối: không có cycle.
Ta không cần kiểm slow.next riêng vì slow không thể vượt fast khi cả hai bắt đầu cùng head
và fast đang được bảo vệ trong trường hợp không có cycle.

In [ ]:
for values in ([],[1],[1,2],[1,2,3]):
    node=linked(values)
    print('độ dài:',len(values),'fast & fast.next?',node is not None and node.next is not None)
    assert baseline(node) is False

In [ ]:
def solve(head: Optional[ListNode]) -> bool:
    slow=fast=head
    while fast is not None and fast.next is not None:
        slow=slow.next
        fast=fast.next.next
        if slow is fast:return True
    return False

### 6.1. Đọc thật chậm từng dòng `slow=slow.next; fast=fast.next.next`

Trong hàm solve, đầu tiên hai biến cùng trỏ head.
Sau mỗi lượt, slow tiến **một**, fast tiến **hai** liên kết.
Sau đó nếu `slow is fast`, chúng gặp cùng object ⇒ có cycle ⇒ trả True.
Nếu điều kiện while không thỏa ở lượt kế, trả False.
Nhớ rằng `slow` và `fast` là **tên biến tham chiếu**, không phải node mới;
phép gán không tạo/chỉnh node nào. Lời giải phải chạy tốt với `head=None`
và list chỉ một node tự trỏ vào chính nó.

In [ ]:
for values,pos in [([], -1),([5],-1),([5],0),([1,2],0)]:
    h=cycle_list(values,pos)
    print('input:',values,'pos:',pos,'đáp án:',solve(h))
    assert solve(h)==(pos!=-1)

### 6.2. Vận tốc tương đối modulo chiều dài vòng

Gọi L là số node trong cycle. Khi cả hai đã ở trong vòng,
mỗi lượt fast nhảy xa hơn slow đúng 1 node tính theo thứ tự chạy.
Nếu khoảng cách hiện tại là `d`, sau một lượt mới là `(d+1)%L`.
Dãy số modulo L chắc chắn sẽ tới 0 trong tối đa L lượt.
Điều này chứng minh cuộc gặp, không phụ thuộc vòng bắt đầu ở đâu;
thời điểm gặp có thể ở một node khác entry.
Thời gian đi trước khi vào vòng cũng tối đa n bước theo `next`,
suy ra tổng thời gian O(n), extra space O(1).

In [ ]:
for L in (1,2,3,5):
    distance=1%L
    visited=[]
    for _ in range(L):
        visited.append(distance)
        distance=(distance+1)%L
    print('vòng dài',L,'khoảng cách tương đối:',visited,'chạm 0?',0 in visited)
    assert 0 in visited

## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Lượt | slow (giá trị) | fast (giá trị) | Hành động |
|---:|---:|---:|---|
| bắt đầu | 3 | 3 | chưa di chuyển |
| 1 | 2 | 0 | chưa gặp |
| 2 | 0 | 2 | chưa gặp |
| 3 | -4 | -4 | **cùng node → True** |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

In [ ]:
def trace_cycle(head):
    slow=fast=head;round_=0
    while fast is not None and fast.next is not None:
        slow=slow.next;fast=fast.next.next;round_+=1
        print(round_,'slow=',slow.val,'fast=',fast.val if fast else None,'cùng node?',slow is fast)
        if slow is fast:return True
    return False
trace_cycle(cycle_list([3,2,0,-4],1))

### 7.1. In bảng trace có identity, không chỉ giá trị

Nếu chỉ in `.val`, hai node trùng giá trị có thể gây hiểu lầm về vị trí gặp.
Để minh họa identity an toàn, ta gắn số thứ tự cho các node ban đầu
bằng cách đi theo `.next` tới hết số node đã biết **trước khi dựng vòng** hoặc
dùng `id(node)` chỉ như công cụ debug; `id()` không cần trong code submit.
Trace ghi slow/fast sau lần di chuyển, rồi dùng `is` để kiểm cuộc gặp.
Nếu fast chạm None, in rõ quyết định trả False; không chạy vô hạn.

In [ ]:
def trace_cycle_identity141(head,limit=20):
    slow=fast=head
    for turn in range(1,limit+1):
        if fast is None or fast.next is None:
            print('Lượt',turn,': fast tới cuối, không có chu trình')
            return False
        slow=slow.next;fast=fast.next.next
        print('Lượt',turn,'slow:',repr(slow),'id=',id(slow),
              'fast:',repr(fast),'id=',id(fast),'gặp?',slow is fast)
        if slow is fast:return True
    raise AssertionError('Chỉ nên demo input nhỏ')
assert trace_cycle_identity141(cycle_list([3,2,0,-4],1))

### 7.2. So sánh cycle tại đầu và cycle nằm sâu trong danh sách

Các vòng có thể bắt đầu ở nút đầu (`pos=0`) hoặc một nút giữa (`pos>0`).
Thuật toán Floyd không cần biết vị trí entry và không cần đọc mảng mẫu.
Đề #141 chỉ yêu cầu **có cycle không**; nếu bài khác yêu cầu entry,
ta phải thêm một giai đoạn riêng như phần biến thể về sau.
Hãy kiểm ba trường hợp: không vòng, vòng tại head, vòng tại nút áp cuối;
đều dùng cùng một chữ ký `solve(head)` và cùng logic slow/fast.

In [ ]:
values=[3,2,0,-4]
for pos in (-1,0,2):
    head0=cycle_list(values,pos)
    print('pos',pos,'->',solve(head0),'baseline',baseline(head0))
    assert solve(head0)==baseline(head0)

### 7.3. Vòng một nút và danh sách có hai nút

Danh sách một nút không vòng `a -> None` phải trả False.
Nếu chính nút đó trỏ về nó `a.next = a`, fast và slow đều đến a sau một lượt,
trả True. Đây là trường hợp rất dễ gây lỗi nếu viết sai điều kiện dừng.
Với hai nút `a -> b -> a`, slow tới b và fast tới a ở lượt đầu,
sau đó slow trở về a và fast cũng a ở lượt hai.
Hãy dùng chính bộ test này để kiểm thứ tự di chuyển/so của code.

In [ ]:
one_no=cycle_list([8],-1)
one_yes=cycle_list([8],0)
two_yes=cycle_list([8,9],0)
for h in (one_no,one_yes,two_yes):
    print('quan sát:',node_values(h,limit=6),'->',solve(h))
assert not solve(one_no) and solve(one_yes) and solve(two_yes)

### 7.4. Không thể chỉ dựa vào số lần lặp trước khi dừng

Một lỗi là gán số vòng lặp cố định như 100 rồi kết luận False nếu chưa gặp.
Nhưng vòng lặp có thể nằm xa sau 100 node; đó là **giả định ngoài đề**.
Floyd không cần biết chiều dài đầu vào, chỉ dựa vào việc fast tới cuối
hoặc slow gặp fast. Thử một danh sách có 200 node dẫn tới cycle gần cuối,
thuật toán vẫn trả True. Không in mọi bước cho ví dụ dài để notebook dễ đọc.
Quan trọng: khi có chu trình, `node_values` chỉ in tối đa `limit`,
tránh vô tình tạo vòng lặp in vô hạn.

In [ ]:
long_cycle=cycle_list(list(range(200)),198)
print('length đầu vào = 200, entry pos=198')
print('baseline:',baseline(long_cycle),'Floyd:',solve(long_cycle))
assert baseline(long_cycle) and solve(long_cycle)

## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Không có chu trình: fast bước 2 node sẽ chạm None. Có chu trình: khi hai con trỏ cùng ở trong vòng, khoảng cách giữa chúng giảm 1 (tính modulo độ dài vòng) sau mỗi lượt, nên cuối cùng chúng gặp đúng cùng node. Do vậy kết quả True/False đúng.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [ ]:
from typing import Optional
class Solution:
    def hasCycle(self, head: Optional[ListNode]) -> bool:
        slow = fast = head
        while fast is not None and fast.next is not None:
            slow = slow.next
            fast = fast.next.next
            if slow is fast:
                return True
        return False

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [ ]:
for values,pos,expected in [([3,2,0,-4],1,True),([1,2],0,True),([1],-1,False),([1],0,True),([], -1,False),([1,2,1],-1,False)]:
    assert solve(cycle_list(values,pos))==expected
    assert Solution().hasCycle(cycle_list(values,pos))==expected
    assert baseline(cycle_list(values,pos))==expected
print('PASS 6 ca chu trình')

### 10.1. Test mọi `pos` hợp lệ với nhiều độ dài ngắn

Một bộ test tốt phải thay đổi cả số node n và vị trí bắt đầu cycle.
Với n node, `pos=-1` nghĩa không có vòng, còn `0..n-1` đều tạo vòng.
Ta liệt kê mọi pos khi n từ 1 tới 25 và kiểm cả baseline lẫn Floyd.
Đây là cách bao phủ vòng ngay đầu, vòng một node, vòng ở gần đuôi và
đường trước vòng dài ngắn khác nhau.
Giá trị node được tạo trùng nhau có chủ đích để kiểm thuật toán
**không phụ thuộc giá trị**; chỉ identity mới quan trọng.

In [ ]:
checked141=0
for n in range(0,26):
    values=[i%3 for i in range(n)]
    for pos in range(-1,n):
        h=cycle_list(values,pos)
        expected=(pos!=-1)
        assert solve(h)==expected and baseline(h)==expected
        assert Solution().hasCycle(h)==expected
        checked141+=1
print('PASS',checked141,'cấu hình n và pos')

### 10.2. Test ngẫu nhiên có seed và so với set oracle

Bằng seed cố định, khi một test lỗi ta có thể tái tạo đúng input.
Tạo n node ngẫu nhiên, chọn pos=-1 hoặc pos hợp lệ,
so sánh `solve`, `baseline` và code LeetCode.
Lý do dùng baseline set làm oracle là nó có chứng minh độc lập:
nó chỉ hỏi có gặp lại **cùng node** hay không, dễ kiểm bằng tay.
Bởi vậy ta tránh kiểm bằng một bản sao khác của chính Floyd,
vốn có thể lặp lại cùng một lỗi.

In [ ]:
import random
rng141=random.Random(141)
for _ in range(600):
    n=rng141.randrange(0,45)
    values=[rng141.randrange(-2,3) for _ in range(n)]
    pos=rng141.randrange(-1,n) if n else -1
    head0=cycle_list(values,pos)
    expected=baseline(head0)
    assert solve(head0)==expected
    assert Solution().hasCycle(head0)==expected
print('PASS 600 test ngẫu nhiên')

### 10.3. Không làm thay đổi các liên kết `.next` của danh sách

Floyd chỉ thay đổi biến `slow` và `fast`, không sửa thuộc tính `.next`
của bất cứ node nào. Đây là một **thuộc tính bất biến của input** quan trọng.
Ta lưu snapshot identity của đường đi tối đa n bước trước và sau khi gọi solve.
Nếu một lời giải lỡ viết `slow.next = fast`, snapshot sẽ thay đổi ngay.
Dùng giới hạn bước vì chuỗi có cycle không thể được duyệt tới None;
snapshot là công cụ kiểm thử, không phải phần code submit.

In [ ]:
def next_snapshot141(head,count):
    p=head;out=[]
    for _ in range(count):
        if p is None:break
        out.append((id(p),id(p.next) if p.next is not None else None))
        p=p.next
    return out
h=cycle_list([10,20,30,40],1)
before=next_snapshot141(h,12)
assert solve(h) is True
after=next_snapshot141(h,12)
assert before==after
print('PASS input không bị thay đổi')

## 11. Time / Space Complexity — vì sao Floyd tuyến tính

Gọi `n` là số node có thể đi tới từ head cho đến khi chạm cuối, hoặc tổng độ dài đoạn trước vòng cộng độ dài vòng.
Baseline set dùng **O(n)** time trung bình và **O(n)** memory để ghi nút đã gặp.
Floyd hai con trỏ: fast đi qua đoạn trước vòng, nếu có vòng thì gặp slow trong không quá O(L) lượt bổ sung (L = chiều dài vòng) ⇒ Time **O(n)**. Chỉ có `slow,fast`, extra Space **O(1)**.
Không cần sửa `.next` của bất cứ node nào, không cấp phát node mới, không nhận `pos` trong chữ ký nộp LeetCode.

## 12. Các lỗi người mới hay mắc

- ❌ So `slow.val==fast.val`: hai node khác nhau có thể cùng giá trị.
- ❌ Đọc `fast.next.next` mà không kiểm tra `fast`/`fast.next`.
- ❌ In linked list có chu trình bằng vòng `while head` không giới hạn sẽ chạy mãi.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

In [ ]:
sample_bug141=linked([7,7,7])
print('3 node khác nhau có cùng value:',node_values(sample_bug141))
print('phải trả False:',solve(sample_bug141))
assert not solve(sample_bug141)

### 12.1. Sai lầm: so `slow.val == fast.val` thay vì `slow is fast`

Nhiều nút khác nhau có thể cùng giá trị, ví dụ 1→1→None.
Khi chạy Floyd một lượt, slow đến nút thứ hai có val=1, fast đến None.
Với input dài hơn 1→2→1→3→None, ở một lượt nào đó slow và fast có thể
cùng val nhưng trỏ đến node khác nhau. Nếu so value sẽ báo cycle sai.
Để sửa, ta cần kiểm **cùng object**, chính là toán tử `is`.
Một test có nhiều val trùng nhưng không có cycle là phản ví dụ bắt buộc.

In [ ]:
def wrong_compare_values141(head):
    slow=fast=head
    while fast is not None and fast.next is not None:
        slow=slow.next;fast=fast.next.next
        if fast is not None and slow.val==fast.val:return True
    return False
no_cycle=linked([1,1,1,1,1])
print('sai vì value:',wrong_compare_values141(no_cycle),'đúng:',solve(no_cycle))
assert wrong_compare_values141(no_cycle) and not solve(no_cycle)

### 12.2. Sai lầm: so trước khi di chuyển và đọc `.next.next` vô điều kiện

Cả slow và fast đều khởi tạo bằng head. So `slow is fast` **trước** khi tiến
sẽ cho True với mọi danh sách không rỗng, kể cả không vòng.
Một lỗi khác là không kiểm `fast.next` trước khi truy cập `fast.next.next`;
trên danh sách một node, nó gây lỗi thuộc tính None.
Cách sửa là `while fast is not None and fast.next is not None:`
rồi di chuyển, **sau đó** mới so identity.
Không cần try/except để giấu lỗi logic; sửa điều kiện trước khi đọc.

In [ ]:
def wrong_check_too_early141(head):
    slow=fast=head
    if slow is fast and head is not None:return True
    return False
one=linked([99])
print('sai:',wrong_check_too_early141(one),'đúng:',solve(one))
assert wrong_check_too_early141(one) and not solve(one)

## 13. Nhận diện dạng bài và vận dụng

Linked list cần phát hiện **vòng lặp** mà không tạo set → slow/fast (Floyd cycle detection).

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Tạo list có **giá trị trùng nhưng node khác nhau** mà không tạo vòng: thuật toán phải trả False.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [ ]:
print('trùng giá trị, không chu trình:',solve(cycle_list([1,2,1,2],-1)))

### 14.1. Biến thể: tìm node BẮT ĐẦU chu trình (bài #142)

Sau khi hai con trỏ gặp nhau, đặt thêm `entry=head` và giữ `slow` ở điểm gặp.
Cho `entry` và `slow` mỗi lượt đi một bước; chúng sẽ gặp tại node đầu vòng.
Đây là định lý Floyd giai đoạn hai. Nếu không có cycle thì trả None.
Chú ý **trả node object**, không trả `.val`, vì có thể có nhiều nút cùng giá trị.
#141 chỉ cần bool; phần tìm entry là biến thể để hiểu thêm cấu trúc thuật toán,
không được thay thế chữ ký `hasCycle` khi nộp bài này.

In [ ]:
def cycle_entry141(head):
    slow=fast=head
    while fast is not None and fast.next is not None:
        slow=slow.next;fast=fast.next.next
        if slow is fast:break
    else:
        return None
    p=head
    while p is not slow:
        p=p.next;slow=slow.next
    return p
a=cycle_list([3,2,0,-4],1)
print('entry value:',cycle_entry141(a).val)
assert cycle_entry141(a) is a.next

### 14.2. Biến thể: đếm độ dài cycle sau khi gặp

Nếu chỉ cần biết có cycle hay không, phép kiểm `slow is fast` đủ.
Nếu muốn biết vòng dài bao nhiêu node, sau cuộc gặp, giữ `slow` cố định,
cho `p=slow.next` lần lượt đi tới khi `p is slow`, đếm số bước.
Mỗi node trên vòng được đi qua đúng một lần, time O(L), extra O(1).
Nếu không có cycle thì trả 0, không cần cấp phát set.
Đây là biến thể hữu ích để kiểm tra hiểu biết về identity và đường đi modulo.

In [ ]:
def cycle_length141(head):
    slow=fast=head
    while fast is not None and fast.next is not None:
        slow=slow.next;fast=fast.next.next
        if slow is fast:
            length=1;p=slow.next
            while p is not slow:
                length+=1;p=p.next
            return length
    return 0
for arr,pos,expected in [([7],0,1),([1,2,3,4],1,3),([1,2,3],-1,0)]:
    assert cycle_length141(cycle_list(arr,pos))==expected
    print('nút:',len(arr),'entry:',pos,'độ dài vòng:',expected)

## 15. Hệ thống luyện tập — 6 bài từ cơ bản đến nâng cao

**Bài 1 — Dự đoán output Python:** Với hai `ListNode(7)` mới tạo, dự đoán `a.val == b.val` và `a is b`.

**Bài 2 — Trace tay:** Trace `cycle_list([3,2,0,-4],1)`, ghi slow/fast sau từng lượt và vị trí đầu tiên chúng cùng object.

**Bài 3 — Hoàn thiện code chính:** tự viết hàm `exercise_cycle141(head)` mà không nhìn cell nộp.

**Bài 4 — Biến thể:** Viết hàm trả ĐỘ DÀI chu trình, hoặc 0 nếu danh sách không có chu trình.

**Bài 5 — Viết test phản ví dụ:** Tự tạo ít nhất 5 ca `(values,pos,expected)` gồm rỗng, tự trỏ 1 node, lặp ở head, lặp ở giữa và nhiều val trùng mà không có cycle.

**Bài 6 — Mini challenge:** Nâng cao: trả CHÍNH NODE bắt đầu chu trình, hoặc None nếu không có; dùng giai đoạn 2 Floyd.

**Cách tự học:** làm bài 1–2 trên giấy → viết TODO bài 3–5 → tự kiểm bằng test của chính mình → thử bài 6. Hãy ghi rõ lý do chọn trạng thái, ranh giới vòng lặp và điều kiện dừng; **đáp án tham khảo nằm sau toàn bộ đề bài**.

### Bài 3 — TODO: tự viết lại Linked List Cycle

Viết code vào hàm ở cell tiếp theo theo đúng kiểu input/output của đề.
Không gọi hàm TODO khi chạy toàn bộ notebook. Khi hoàn thành, tự bỏ `raise NotImplementedError` rồi chạy test được gợi ý dưới hàm.
**Tiêu chí tự chấm:** giải thích được vì sao code đúng ngay cả ở trường hợp nhỏ nhất, và không làm thao tác dư thừa so với lời giải chính.

In [ ]:
def exercise_cycle141(head):
    # TODO 1: slow = fast = head
    # TODO 2: bảo vệ fast và fast.next trước khi nhảy 2 bước
    # TODO 3: sau khi di chuyển, so slow is fast
    # TODO 4: trả đúng boolean
    raise NotImplementedError("Tự viết trước khi test")
# assert exercise_cycle141(cycle_list([1,2],0)) is True

### Bài 4 — TODO: đổi yêu cầu, giữ kỹ thuật cốt lõi

Viết hàm trả ĐỘ DÀI chu trình, hoặc 0 nếu danh sách không có chu trình.

Đừng chép y nguyên code submit: hãy chỉ ra đầu ra thay đổi ở đâu; tự tạo 2 input mới để kiểm thử.

In [ ]:
def exercise_cycle_length141(head):
    # TODO: dùng Floyd tìm một điểm gặp.
    # Nếu có gặp, đi trọn vòng bằng một pointer và đếm số node.
    raise NotImplementedError("Chưa làm bài 4")
# assert exercise_cycle_length141(cycle_list([1,2,3],1)) == 2

### Bài 5 — TODO: tạo bộ kiểm thử có chủ đích

Tự tạo ít nhất 5 ca `(values,pos,expected)` gồm rỗng, tự trỏ 1 node, lặp ở head, lặp ở giữa và nhiều val trùng mà không có cycle.

Mỗi test phải có **expected tự tính trước**, không được lấy `expected=solve(input)`. Một test tốt thường có mục tiêu bắt một kiểu sai cụ thể. Nếu có node, dựng dữ liệu mới ở mỗi lượt test.

In [ ]:
def exercise_cases141():
    # TODO: trả [(values,pos,expected_bool), ...] ít nhất 5 trường hợp.
    # KHÔNG trả list các ListNode trực tiếp; để hàm dựng input riêng.
    raise NotImplementedError("Chưa làm bài 5")

### Bài 6 — Mini challenge chuyển giao kỹ thuật (khó)

**Yêu cầu:** Nâng cao: trả CHÍNH NODE bắt đầu chu trình, hoặc None nếu không có; dùng giai đoạn 2 Floyd.

    Bài 6 không thuộc đề gốc. Hãy tự phân tích **miền input, output, tính đúng**
    và chọn một phản ví dụ nhỏ trước khi code. Nếu thay đổi cấu trúc lời giải,
    cần nói rõ điều kiện nào đã thay đổi so với bài gốc.
    Không gọi hàm có TODO trong lần chạy toàn bộ notebook: để nguyên `NotImplementedError`
    cho tới khi bạn tự điền mã và tự bật test.
    **Kiểm tra:** so kết quả với đáp án cuối, nhưng chỉ sau khi đã tự làm.

In [ ]:
def exercise_cycle_entry141(head):
    # TODO 1: chạy slow/fast tới điểm gặp; nếu không gặp -> None.
    # TODO 2: p=head, q=điểm gặp; cùng đi mỗi lần 1 bước.
    # TODO 3: khi p is q thì return chính object node ấy.
    raise NotImplementedError("Mini challenge chưa giải")
# h=cycle_list([3,2,0,-4],1); assert exercise_cycle_entry141(h) is h.next

## 16. Gợi ý theo ba cấp — chỉ đọc sau khi tự thử

**Cấp 1:** cycle nghĩa gặp lại **cùng object**, không phải trùng giá trị.
**Cấp 2:** `while fast is not None and fast.next is not None` rồi slow đi 1, fast đi 2.
**Cấp 3:** So identity sau khi di chuyển. Nếu gặp trong vòng, độ dài vòng là số lượt `p=p.next` để quay lại điểm gặp.
**Nâng cao:** Để tìm entry, sau khi gặp đặt một pointer tại head và một pointer tại điểm gặp, cùng tiến mỗi lượt 1 nút.

**Cách sử dụng gợi ý:** bắt đầu ở cấp 1; nếu vẫn chưa ra, chỉ mở cấp 2 rồi tự viết tiếp. Không đọc lời giải đầy đủ ngay khi vừa gặp khó. Với bài 6, thử tự xác định input nhỏ nhất gây sai trong phương án bạn đề xuất.

## 17. Đáp án tham khảo — chỉ đọc sau khi đã làm TODO

Các hàm **đáp án** dùng tên khác với `exercise_*` để không vô tình thực thi code chưa hoàn thiện.
Đáp án bài 3 được định nghĩa trước, rồi đến các biến thể bài 4–6 và bộ test tương ứng.
Sau khi chạy, hãy so sánh logic từng dòng với lời giải chính và tìm lý do các cài đặt tương đương, không chỉ so con số output.

In [ ]:
def exercise_core_answer(head):
    slow=fast=head
    while fast is not None and fast.next is not None:
        slow=slow.next
        fast=fast.next.next
        if slow is fast:return True
    return False

print("Đã định nghĩa đáp án tham khảo; xem lại từng dòng rồi tự gõ lại.")

### Đáp án bài 4 — biến thể với yêu cầu đầu ra mới

Bài 4 yêu cầu: Viết hàm trả ĐỘ DÀI chu trình, hoặc 0 nếu danh sách không có chu trình.

    So với lời giải gốc, hãy chỉ ra phần nào giữ nguyên:
    cú pháp duyệt, trạng thái trước/sau, bất biến. Phần thay đổi phải tương ứng
    đúng với **đầu ra mới**, không làm thay chữ ký code submit #
    hoặc vô tình lẫn thêm điều kiện ngoài phạm vi bài gốc.
    Đoạn đáp án dưới đây có assert riêng để kiểm tính đúng.

In [ ]:
def answer_cycle_length141(head):
    slow=fast=head
    while fast is not None and fast.next is not None:
        slow=slow.next;fast=fast.next.next
        if slow is fast:
            length=1;p=slow.next
            while p is not slow:
                p=p.next;length+=1
            return length
    return 0
for values,pos,expected in [([], -1,0),([7],0,1),([1,2,3],1,2),([1,2,3],-1,0)]:
    assert answer_cycle_length141(cycle_list(values,pos))==expected
print('PASS đáp án bài 4 — độ dài vòng')

### Đáp án bài 6 và kiểm thử đáp án bài 5

**Bài 6:** Nâng cao: trả CHÍNH NODE bắt đầu chu trình, hoặc None nếu không có; dùng giai đoạn 2 Floyd.

    Đọc suy luận trước khi so code: giữ trạng thái nào; tại mỗi bước
    trạng thái cập nhật ra sao; vì sao khi dừng, kết quả đáp ứng yêu cầu.
    Bài 5 là tự xây bộ test: đáp án tham khảo còn có các mẫu độc lập,
    chạy kèm assert nhưng **không hề chạy TODO của người học**.
    Nếu bộ test bạn tự nghĩ khác, điều đó bình thường: hãy đảm bảo cả
    test của bạn và test tham khảo đều có expected đúng.

In [ ]:
def answer_cycle_entry141(head):
    slow=fast=head
    while fast is not None and fast.next is not None:
        slow=slow.next;fast=fast.next.next
        if slow is fast:break
    else:return None
    p=head
    while p is not slow:
        p=p.next;slow=slow.next
    return p
for values,pos in [([3,2,0,-4],1),([1,2],0),([1],0),([1,2,3],-1)]:
    h=cycle_list(values,pos)
    node=answer_cycle_entry141(h)
    if pos==-1:assert node is None
    else:
        target=h
        for _ in range(pos):target=target.next
        assert node is target
print('PASS đáp án mini challenge tìm entry')

cases141=[([], -1,False),([7],-1,False),([7],0,True),([1,2],0,True),([1,2,3,4],2,True),([1,2,1,2],-1,False)]
for values,pos,expected in cases141:
    assert solve(cycle_list(values,pos))==expected
print('PASS 6 ca test identity')

### Lời giải thích và đáp án ngắn cho cả 6 bài

**Bài 1:** hai node val bằng nhau nhưng `a is b` False vì khác object.
**Bài 2:** hãy đối chiếu bảng của `trace_cycle`/`trace_cycle_identity141`: slow/fast gặp nhau sau khi cả hai đã di chuyển.
**Bài 3:** `exercise_core_answer` triển khai Floyd trả bool, không nhận `pos`.
**Bài 4:** `answer_cycle_length141` đi thêm một vòng để đếm L.
**Bài 5:** bộ test bao gồm node trùng value không vòng, rất quan trọng để bác bỏ lối dùng `node.val`.
**Bài 6:** `answer_cycle_entry141` trả về CHÍNH nút bắt đầu vòng bằng toán tử `is`, không chỉ `.val`.

Đối với bài tập viết code, hãy so sánh cả **điều kiện lặp**, **thứ tự cập nhật** và **cách return** chứ không chỉ so giá trị cuối. Các hàm `answer_*` của bài 4 và 6 nằm ở những cell ngay phía trên để người học có thể chạy riêng sau khi tự làm.

## 18. Tóm tắt và checklist trước khi nộp

**Tư duy phải tự diễn đạt được bằng lời:** nêu bài toán bằng một câu, chỉ rõ trạng thái nào đủ để quyết định bước kế tiếp, rồi mới viết code. Đọc lại các ví dụ khó trong notebook; khi một điều kiện `if` làm bạn phân vân, hãy lập bảng giá trị **ngay trước và sau** câu lệnh ấy.

1. Tôi đã phân biệt đầu vào/đầu ra, và đọc đúng chữ ký `class Solution` của bài.
2. Tôi đã tự chạy thử ví dụ nhỏ, biên, phản ví dụ và kiểm thử đối chiếu nhiều cách giải.
3. Tôi hiểu từng cấu trúc Python, có thể giải thích vì sao dùng nó thay vì cách khác.
4. Tôi chỉ ra được bất biến: sau bước thứ k, điều gì **luôn đúng** và tại sao?
5. Tôi phân tích Time và Space dựa trên số lượng phần tử/nút/bít và có tính tới output/stack/chuỗi tạm.
6. Tôi đã hoàn thành TODO **trước** khi kéo tới đáp án tham khảo.
7. Tôi chỉ sao chép `class Solution` để nộp LeetCode; các hàm dựng input, trace và bộ test dành cho notebook, không dành cho submit.

**Tự đánh giá:** bạn hiểu bài khi tự viết được lời giải trên một trang trống và xử lý được một biến thể mà không sao chép mã cũ.

**Mẫu nhận dạng của bài 141:** chuỗi các node theo `.next` có thể tạo vòng → Floyd hai vận tốc, so tham chiếu `is`, O(n) time, O(1) extra space.